# 04 — Per-technician analysis

How the agent distributed work across the fleet, and how each
technician's knowledge / fatigue / specialisation evolved.

In [ ]:
import sys, pathlib
# Make ``utils.py`` importable whether the notebook is opened from
# project root or from analysis/.
_here = pathlib.Path.cwd()
for cand in (_here, _here / "analysis", _here.parent):
    if (cand / "utils.py").exists():
        sys.path.insert(0, str(cand))
        break

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils import (
    list_runs, latest_run, load_run, load_runs,
    group_columns, group_columns_by_prefix, strip_prefix, rolling_mean,
)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)

In [ ]:
# Pick a run.  ``latest_run()`` returns the most recently modified
# experiment in ``reports/``.  Override ``EXP_ID = "..."`` to analyse
# a specific one.
EXP_ID = latest_run()
print("Analysing:", EXP_ID)

ep, st, cfg = load_run(EXP_ID)
print(f"  episode_metrics.csv: shape={ep.shape}")
print(f"  step_metrics.csv:    shape={st.shape if st is not None else 'absent'}")

## Total assignments per technician across the run

In [ ]:
assign_cols = group_columns(ep, "assignments")
if assign_cols:
    totals = ep[assign_cols].sum().sort_values(ascending=False)
    totals.index = strip_prefix(list(totals.index), "assignments")
    fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * len(totals))))
    totals.plot.barh(ax=ax, color="C0", edgecolor="black", linewidth=0.5)
    ax.invert_yaxis()
    ax.set(xlabel="repairs assigned", title="Workload distribution across the fleet")
    plt.show()
else:
    print("No assignments/* columns in this run.")

## Assignment-share evolution (training only)

A line per technician showing what *fraction* of an episode's repairs
went to them.  Drift towards 1 / n_techs ⇒ uniform; spikes ⇒
favouritism.

In [ ]:
train = ep[ep["phase"] == "train"]
if assign_cols and not train.empty:
    counts = train[assign_cols].copy()
    counts.columns = strip_prefix(list(counts.columns), "assignments")
    shares = counts.div(counts.sum(axis=1).replace(0, 1), axis=0)
    shares["episode"] = train["episode"].values

    fig, ax = plt.subplots()
    for col in shares.columns[:-1]:
        ax.plot(shares["episode"], rolling_mean(shares[col]), label=col, lw=1.5)
    ax.axhline(1.0 / max(1, len(shares.columns) - 1), ls="--", color="grey", lw=0.8,
               label="uniform")
    ax.set(xlabel="episode", ylabel="share of repairs", title="Assignment share per tech (rolling)")
    ax.legend(fontsize=8, ncol=2)
    plt.show()

## Per-tech knowledge / fatigue evolution (step level)

If step metrics are available, plot mean ``tech_knowledge/<t>`` and
``tech_fatigue/<t>`` over the steps of the last training episode.

In [ ]:
if st is not None and not st.empty:
    last_ep = st[(st["phase"] == "train") & (st["episode"] == st[st["phase"] == "train"]["episode"].max())]
    if last_ep.empty:
        last_ep = st[st["episode"] == st["episode"].max()]
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    for kind, ax in zip(("tech_knowledge", "tech_fatigue"), axes):
        cols = group_columns(last_ep, kind)
        for col in cols:
            ax.plot(last_ep["step"], last_ep[col], label=col.split('/', 1)[1], lw=1.2)
        ax.set(xlabel="step within episode", ylabel=kind,
               title=f"{kind} on episode {int(last_ep['episode'].iloc[0])}")
        ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print("No step metrics available.")

## Specialisation — what component types each tech handled

Cross-tab of (assigned tech) × (ticket_component_type) from the step
log, normalised per row so you can see relative specialisation.

In [ ]:
if st is not None and not st.empty and "ticket_component_type" in st.columns:
    train_steps = st[st["phase"] == "train"]
    if not train_steps.empty:
        # Action index → no per-tech name in step log; key on action int
        ct = pd.crosstab(train_steps["action"], train_steps["ticket_component_type"])
        # Drop empty component-type column if any
        ct = ct.loc[:, ct.sum() > 0]
        norm = ct.div(ct.sum(axis=1).replace(0, 1), axis=0)
        fig, ax = plt.subplots(figsize=(min(12, 1 + 0.8 * ct.shape[1]), 0.4 * ct.shape[0] + 1))
        im = ax.imshow(norm.values, aspect="auto", cmap="viridis")
        ax.set_xticks(range(ct.shape[1])); ax.set_xticklabels(ct.columns, rotation=30, ha="right")
        ax.set_yticks(range(ct.shape[0])); ax.set_yticklabels(["tech_" + str(i) for i in ct.index])
        ax.set_title("Specialisation — % of each tech's repairs by component")
        plt.colorbar(im, ax=ax, label="row-normalised share")
        plt.tight_layout(); plt.show()